<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/16_metadata_filtering_rag/metadata_filtering_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    {"text": "Paris is the capital of France.", "category": "geography"},
    {"text": "Berlin is the capital of Germany.", "category": "geography"},
    {"text": "Python is a programming language.", "category": "technology"},
    {"text": "Machine learning is a subset of AI.", "category": "technology"}
]

In [11]:
query_category = "geography"

filtered_docs = [doc["text"] for doc in documents if doc["category"] == query_category]

print("FILTERED DOCUMENTS:")
for doc in filtered_docs:
    print("-", doc)

FILTERED DOCUMENTS:
- Paris is the capital of France.
- Berlin is the capital of Germany.


In [ ]:
import faiss
import numpy as np

embeddings = embed_model.encode(filtered_docs)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [ ]:
query = "What is the capital of France?"

query_embedding = embed_model.encode([query])

In [9]:
D, I = index.search(np.array(query_embedding), k=1)

retrieved_doc = filtered_docs[I[0][0]]

print("RETRIEVED DOCUMENT:")
print(retrieved_doc)

RETRIEVED DOCUMENT:
Paris is the capital of France.


In [10]:
prompt = f"""
Use the following context to answer the question.

Context: {retrieved_doc}

Question: {query}
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=30)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nFINAL ANSWER:")
print(answer)


FINAL ANSWER:
Paris
